# EDA - UCI Adult Census Income (1994)

Four questions, asked identically in `02_acs_pums_eda.ipynb` so the two datasets are comparable:

1. Where is data missing, and is missingness related to the protected attributes?
2. Which features act as proxies for the protected attributes?
3. How large is the income gap by sex, and what explains it?
4. How large is the income gap by race, and how should race be grouped for the two-group metrics?

## 0. Setup

Helper functions are defined here so the same calculation is applied to both datasets.

In [1]:
import os
from itertools import combinations

import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency, norm

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)

RANDOM_SEED = 0
RESULTS_DIR = "../results/tables"
os.makedirs(RESULTS_DIR, exist_ok=True)


def cramers_v(confusion_matrix):
    """Cramér's V used to measure how strongly two categorical columns are associated."""
    chi2 = chi2_contingency(confusion_matrix, correction=False)[0]
    n = confusion_matrix.sum().sum()
    r, k = confusion_matrix.shape
    return (chi2 / (n * (min(r, k) - 1))) ** 0.5


def two_proportion_test(count1, n1, count2, n2):
    """Two-proportion z-test used to compare two groups on a yes/no outcome."""
    p1, p2 = count1 / n1, count2 / n2
    p_pool = (count1 + count2) / (n1 + n2)
    se = (p_pool * (1 - p_pool) * (1 / n1 + 1 / n2)) ** 0.5
    z = (p1 - p2) / se
    return p1, p2, 2 * (1 - norm.cdf(abs(z)))


def pairwise_proportion_tests(df, group_col, outcome_col, positive_value):
    """two-proportion z-test on every pair of groups. One row is returned per pair."""
    counts = pd.crosstab(df[group_col].astype(str), df[outcome_col].astype(str))
    counts["n"] = counts.sum(axis=1)

    rows = []
    for g1, g2 in combinations(counts.index, 2):
        p1, p2, p = two_proportion_test(
            count1=counts.loc[g1, positive_value], n1=counts.loc[g1, "n"],
            count2=counts.loc[g2, positive_value], n2=counts.loc[g2, "n"],
        )
        rows.append({
            "group_1": g1, "rate_1_pct": round(p1 * 100, 2), "n_1": counts.loc[g1, "n"],
            "group_2": g2, "rate_2_pct": round(p2 * 100, 2), "n_2": counts.loc[g2, "n"],
            "p_value": p, "verdict": "DIFFERENT" if p < 0.05 else "same",
        })
    return pd.DataFrame(rows).sort_values("p_value").reset_index(drop=True)


def proxy_screen(df, protected_col, feature_cols):
    """Measures how strongly each feature is associated with a protected attribute."""
    scores = {
        col: cramers_v(pd.crosstab(df[col], df[protected_col]))
        for col in feature_cols if col != protected_col
    }
    return pd.Series(scores).sort_values(ascending=False).round(3)


def group_rates(df, group_col, outcome_col, positive_value):
    """Percentage of each group receiving the positive outcome."""
    return pd.crosstab(df[group_col], df[outcome_col], normalize="index")[positive_value] * 100


def controlled_gap(df, control_col, group_col, outcome_col, positive_value,
                   advantaged, disadvantaged, min_group=0):
    """Compare two groups within levels of a control variable.

Measures the group gap separately within each control level, then averages
across levels weighted by record count. If the weighted gap is much smaller
than the raw gap, the control explains the confound.

Levels with fewer than `min_group` records in either group are dropped as
unreliable. Returns the per-level breakdown and the overall weighted gap.
"""
    table = (
        df.groupby([control_col, group_col], observed=True)[outcome_col]
        .apply(lambda s: (s == positive_value).mean() * 100)
        .unstack(group_col)
    )
    counts = pd.crosstab(df[control_col], df[group_col]).reindex(table.index)

    table["gap"] = table[advantaged] - table[disadvantaged]
    table["n"] = counts.sum(axis=1)
    table["smallest_group"] = counts[[advantaged, disadvantaged]].min(axis=1)

    used = table[table["smallest_group"] >= min_group].dropna(subset=["gap"])
    weighted = (used["gap"] * used["n"]).sum() / used["n"].sum()

    return table, weighted

In [2]:
DATASET = "UCI Adult (1994)"
POSITIVE = ">50K"

## 1. Data loading

Accessed via `fetch_openml` for reproducibility from code alone; returns 48,842 records, matching the published dataset size. The OpenML target column `class` is renamed to `income`.

In [3]:
from sklearn.datasets import fetch_openml

adult = fetch_openml(name="adult", version=2, as_frame=True)
df_uci = adult.frame.rename(columns={"class": "income"})

print("shape:", df_uci.shape)
df_uci.head()

shape: (48842, 15)


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K
4,18,NaN,103497,Some-college,10,Never-married,NaN,Own-child,White,Female,0,0,30,United-States,<=50K


In [4]:
print(df_uci.dtypes)
print()
print(df_uci["income"].value_counts())
print()
print((df_uci["income"].value_counts(normalize=True) * 100).round(2))

age                  int64
workclass         category
fnlwgt               int64
education         category
education-num        int64
marital-status    category
occupation        category
relationship      category
race              category
sex               category
capital-gain         int64
capital-loss         int64
hours-per-week       int64
native-country    category
income            category
dtype: object

income
<=50K    37155
>50K     11687
Name: count, dtype: int64

income
<=50K    76.07
>50K     23.93
Name: proportion, dtype: float64


### Findings: dataset shape

- 48,842 records and 15 columns are returned, which matches the published size of the
  dataset.
- The target is imbalanced. 76.1% of records fall in the `<=50K` class and 23.9% in the
  `>50K` class.


## 2. Missing values

`fetch_openml` returns missing values as nulls (recorded as `"?"` in the original UCI file).

In [5]:
missing = df_uci.isnull().sum()
missing = missing[missing > 0]

print(missing)
print()
print((missing / len(df_uci) * 100).round(2))

workclass         2799
occupation        2809
native-country     857
dtype: int64

workclass         5.73
occupation        5.75
native-country    1.75
dtype: float64


In [6]:
both_missing = df_uci["workclass"].isnull() & df_uci["occupation"].isnull()
only_workclass = df_uci["workclass"].isnull() & df_uci["occupation"].notnull()
only_occupation = df_uci["occupation"].isnull() & df_uci["workclass"].notnull()

print("both missing:", both_missing.sum())
print("only workclass missing:", only_workclass.sum())
print("only occupation missing:", only_occupation.sum())

both missing: 2799
only workclass missing: 0
only occupation missing: 10


In [7]:
df_uci[only_occupation]["workclass"].value_counts()

workclass
Never-worked        10
Federal-gov          0
Local-gov            0
Private              0
Self-emp-inc         0
Self-emp-not-inc     0
State-gov            0
Without-pay          0
Name: count, dtype: int64

In [8]:
print(df_uci.groupby("race", observed=True)["native-country"].apply(lambda s: s.isnull().mean() * 100).round(2))
print()
print(df_uci.groupby("sex", observed=True)["native-country"].apply(lambda s: s.isnull().mean() * 100).round(2))

race
Amer-Indian-Eskimo    0.00
Asian-Pac-Islander    8.43
Black                 2.82
Other                 5.91
White                 1.37
Name: native-country, dtype: float64

sex
Female    1.53
Male      1.87
Name: native-country, dtype: float64


In [9]:
ct_missing_race = pd.crosstab(df_uci["race"], df_uci["native-country"].isnull())
print(ct_missing_race)
print()

chi2, p, dof, expected = chi2_contingency(ct_missing_race)
print("chi-square p-value:", p)
print("Cramér's V:", round(cramers_v(ct_missing_race), 4))

native-country      False  True 
race                            
Amer-Indian-Eskimo    470      0
Asian-Pac-Islander   1391    128
Black                4553    132
Other                 382     24
White               41189    573

chi-square p-value: 1.5994525120690708e-108
Cramér's V: 0.1019


### Findings: missing values

- Only 3 of the 15 columns contain missing values: `workclass` (2,799 records, 5.73%),
  `occupation` (2,809 records, 5.75%) and `native-country` (857 records, 1.75%).
- The missingness in `workclass` and `occupation` is not independent. Every record with a
  missing `workclass` also has a missing `occupation`. The 10 records where only
  `occupation` is missing are all recorded as `workclass = "Never-worked"`, which explains
  the pattern: where no job is held, no occupation is reported.
- Missingness in `native-country` is not spread evenly across race. The difference is
  significant (chi-square p < 0.001) and is caused almost entirely by the
  `Asian-Pac-Islander` group, at 8.43% missing against 0% to 5.91% for the other groups.
  The effect size is small (Cramér's V = 0.102), so the association is real but not large.
- Missingness in `native-country` does not vary meaningfully by sex (1.53% against 1.87%).

The uneven spread across race is the evidence behind the handling decision recorded below.

### Decision - missing values are kept as an explicit category

Missing values are retained as an explicit `"Unknown"` category rather than dropped or
imputed, for two reasons:

- **Dropping** the affected records would remove 2,799 rows, and the missingness was shown
  above to fall unevenly across race. The records most relevant to a fairness audit would
  be precisely the ones discarded, biasing the analysis.
- **Imputing** was considered and rejected: filling in a plausible value risks manufacturing
  a relationship between the imputed feature and the protected attributes that does not exist
  in the real non-response pattern.

Treating missingness as its own category preserves every record and keeps the non-response
pattern itself visible to the analysis.

In [10]:
for col in ["workclass", "occupation", "native-country"]:
    if isinstance(df_uci[col].dtype, pd.CategoricalDtype):
        df_uci[col] = df_uci[col].cat.add_categories("Unknown")
    df_uci[col] = df_uci[col].fillna("Unknown")

print("remaining nulls:", df_uci.isnull().sum().sum())
print()
print(df_uci["workclass"].value_counts())

remaining nulls: 0

workclass
Private             33906
Self-emp-not-inc     3862
Local-gov            3136
Unknown              2799
State-gov            1981
Self-emp-inc         1695
Federal-gov          1432
Without-pay            21
Never-worked           10
Name: count, dtype: int64


## 3. Proxy attributes

A removed protected attribute can be rebuilt by a model from correlated *proxy* features, the mechanism of indirect discrimination. Cramér's V measures association strength (below 0.1 negligible, 0.1–0.3 weak, 0.3–0.5 moderate, above 0.5 strong).

In [11]:
feature_cols = [
    "workclass", "education", "marital-status", "occupation", "relationship",
    "race", "sex", "native-country", "income",
]

proxy_sex = proxy_screen(df_uci, "sex", feature_cols)
proxy_race = proxy_screen(df_uci, "race", feature_cols)

print("association with sex:")
print(proxy_sex)
print()
print("association with race:")
print(proxy_race)

association with sex:
relationship      0.647
marital-status    0.459
occupation        0.424
income            0.215
workclass         0.152
race              0.114
education         0.093
native-country    0.061
dtype: float64

association with race:
native-country    0.402
sex               0.114
income            0.100
relationship      0.098
marital-status    0.083
occupation        0.079
education         0.073
workclass         0.058
dtype: float64


In [12]:
print((pd.crosstab(df_uci["relationship"], df_uci["sex"], normalize="index") * 100).round(2))
print()
print(pd.crosstab(df_uci["relationship"], df_uci["sex"]))

sex             Female   Male
relationship                 
Husband           0.01  99.99
Not-in-family    46.65  53.35
Other-relative   45.75  54.25
Own-child        44.53  55.47
Unmarried        76.64  23.36
Wife             99.87   0.13

sex             Female   Male
relationship                 
Husband              1  19715
Not-in-family     5870   6713
Other-relative     689    817
Own-child         3376   4205
Unmarried         3928   1197
Wife              2328      3


In [13]:
(pd.crosstab(df_uci["race"], df_uci["native-country"] == "United-States", normalize="index") * 100).round(2)

native-country,False,True
race,,
Amer-Indian-Eskimo,3.83,96.17
Asian-Pac-Islander,71.76,28.24
Black,8.88,91.12
Other,53.45,46.55
White,7.83,92.17


### Findings : proxy attributes

**Proxies for `sex`:**

| Feature | Cramér's V | Strength |
|---|---|---|
| relationship | 0.647 | strong |
| marital-status | 0.459 | moderate |
| occupation | 0.435 | moderate |
| income (target) | 0.215 | weak |
| workclass | 0.143 | weak |
| race | 0.114 | weak |
| education | 0.093 | negligible |
| native-country | 0.060 | negligible |

- `relationship` is close to a direct restatement of `sex`. The association comes from the
  `Husband` and `Wife` categories, which are almost perfectly sex-determined by definition:
  just 1 woman is recorded as `Husband` and 3 men as `Wife`. The remaining categories are
  close to evenly split by sex and contribute almost nothing to the association.
- `marital-status` and `occupation` are moderate proxies, and would be expected to carry sex
  information into a model even once `sex` itself has been removed.

**Proxies for `race`:**

| Feature | Cramér's V | Strength |
|---|---|---|
| native-country | 0.415 | moderate |
| sex | 0.114 | weak |
| income (target) | 0.100 | negligible |
| relationship | 0.098 | negligible |
| marital-status | 0.083 | negligible |
| occupation | 0.081 | negligible |
| education | 0.073 | negligible |
| workclass | 0.059 | negligible |

- `native-country` is the only meaningful proxy for race, reflecting differing immigration
  histories across these groups: 92.2% of the `White` group and 96.2% of the
  `Amer-Indian-Eskimo` group are US-born, against 28.2% of the `Asian-Pac-Islander` group
  and 46.6% of the `Other` group.
- No feature approaches the strength `relationship` shows for sex, so race is less directly
  recoverable from the remaining features than sex is.

**What this means for the experiments:** in the protected-attribute-removed experiment,
`relationship` should be removed alongside `sex`, and `native-country` alongside `race`.
Removing the protected attribute on its own would leave this information recoverable through
its proxies.

## 4. Income gap by sex

The raw gap is measured, then tested against explanations by comparing male and female rates *within* groups of similar records. If a control explains the gap, the gap shrinks once only similar records are compared.

In [14]:
rates_sex = group_rates(df_uci, "sex", "income", POSITIVE)
raw_gap_sex = rates_sex["Male"] - rates_sex["Female"]

print(rates_sex.round(2))
print()

ct_income_sex = pd.crosstab(df_uci["sex"], df_uci["income"])
chi2, p, dof, expected = chi2_contingency(ct_income_sex)
print("chi-square p-value:", p)
print("Cramér's V:", round(cramers_v(ct_income_sex), 4))
print()
print("raw gap (percentage points):", round(raw_gap_sex, 2))
print("ratio:", round(rates_sex["Male"] / rates_sex["Female"], 2))

sex
Female    10.93
Male      30.38
Name: >50K, dtype: float64

chi-square p-value: 0.0
Cramér's V: 0.2146

raw gap (percentage points): 19.45
ratio: 2.78


### Control 1 - hours worked

In [15]:
print(df_uci.groupby("sex", observed=True)["hours-per-week"].mean().round(2))

sex
Female    36.40
Male      42.42
Name: hours-per-week, dtype: float64


In [16]:
df_uci["hours_bucket"] = pd.cut(df_uci["hours-per-week"], bins=[0, 20, 35, 40, 45, 50, 100])

hours_table, gap_hours = controlled_gap(
    df_uci, control_col="hours_bucket", group_col="sex", outcome_col="income",
    positive_value=POSITIVE, advantaged="Male", disadvantaged="Female",
)

print(hours_table.round(2))
print()
print("raw gap:", round(raw_gap_sex, 2))
print("gap after controlling for hours:", round(gap_hours, 2))
print("% of raw gap remaining:", round(gap_hours / raw_gap_sex * 100, 1))

sex           Female   Male    gap      n  smallest_group
hours_bucket                                             
(0, 20]         5.95   7.70   1.75   4453            2065
(20, 35]        7.65  11.59   3.94   5879            2873
(35, 40]       10.15  26.59  16.45  24158            8170
(40, 45]       16.25  41.01  24.76   3651             837
(45, 50]       26.52  45.67  19.15   5266             920
(50, 100]      21.58  44.59  23.00   5435             871

raw gap: 19.45
gap after controlling for hours: 15.25
% of raw gap remaining: 78.4


### Control 2 - occupation

In [17]:
occ_table, gap_occ = controlled_gap(
    df_uci, control_col="occupation", group_col="sex", outcome_col="income",
    positive_value=POSITIVE, advantaged="Male", disadvantaged="Female",
)

print(occ_table.sort_values("gap", ascending=False).round(2))
print()
print("gap after controlling for occupation:", round(gap_occ, 2))
print("% of raw gap remaining:", round(gap_occ / raw_gap_sex * 100, 1))

sex                Female   Male    gap     n  smallest_group
occupation                                                   
Exec-managerial     24.08  57.33  33.25  6086            1748
Sales                6.88  37.70  30.82  5504            1947
Prof-specialty      25.96  56.03  30.07  6172            2242
Tech-support        11.92  39.93  28.01  1446             562
Protective-serv     12.30  34.03  21.74   983             122
Adm-clerical         8.20  24.92  16.72  5611            1842
Craft-repair        10.22  23.32  13.10  6112             323
Machine-op-inspct    3.48  15.51  12.03  3022             804
Transport-moving    10.24  21.01  10.77  2355             127
Farming-fishing      3.16  12.19   9.03  1490              95
Unknown              5.89  12.37   6.48  2809            1273
Handlers-cleaners    3.15   7.15   4.00  2072             254
Other-service        2.89   5.66   2.77  4923            2225
Priv-house-serv      1.32   0.00  -1.32   242              14
Armed-Fo

### Control 3 - household relationship

Tested last, as `relationship` was the strongest sex proxy above.

In [18]:
rel_table, gap_rel = controlled_gap(
    df_uci, control_col="relationship", group_col="sex", outcome_col="income",
    positive_value=POSITIVE, advantaged="Male", disadvantaged="Female",
)

print(rel_table.sort_values("gap", ascending=False).round(2))
print()
print("gap after controlling for relationship:", round(gap_rel, 2))
print("% of raw gap remaining:", round(gap_rel / raw_gap_sex * 100, 1))

sex             Female   Male    gap      n  smallest_group
relationship                                               
Husband           0.00  44.87  44.87  19716               1
Unmarried         4.12  12.28   8.16   5125            1197
Not-in-family     7.67  12.30   4.64  12583            5870
Own-child         1.21   1.66   0.45   7581            3376
Other-relative    3.48   3.43  -0.06   1506             689
Wife             46.91  33.33 -13.57   2331               3

gap after controlling for relationship: 19.58
% of raw gap remaining: 100.7


The result above is not reliable, and the reason is a data problem rather than a finding.
The `smallest_group` column shows how many records of the smaller sex fall in each category.
The `Husband` category holds 1 woman against 19,716 men, and the `Wife` category holds 3 men
against 2,331 women. A male-against-female comparison cannot be supported inside either
category. Because these are also the two largest categories, the meaningless values they
produce dominate the weighted average.

The calculation is run again with those categories excluded, using the `min_group` argument.

In [19]:
rel_table_fixed, gap_rel_fixed = controlled_gap(
    df_uci, control_col="relationship", group_col="sex", outcome_col="income",
    positive_value=POSITIVE, advantaged="Male", disadvantaged="Female", min_group=30,
)

print(rel_table_fixed[rel_table_fixed["smallest_group"] >= 30].round(2))
print()
print("excluded:", rel_table_fixed[rel_table_fixed["smallest_group"] < 30].index.tolist())
print()
print("gap after controlling for relationship:", round(gap_rel_fixed, 2))
print("% of raw gap remaining:", round(gap_rel_fixed / raw_gap_sex * 100, 1))

sex             Female   Male   gap      n  smallest_group
relationship                                              
Not-in-family     7.67  12.30  4.64  12583            5870
Other-relative    3.48   3.43 -0.06   1506             689
Own-child         1.21   1.66  0.45   7581            3376
Unmarried         4.12  12.28  8.16   5125            1197

excluded: ['Husband', 'Wife']

gap after controlling for relationship: 3.86
% of raw gap remaining: 19.9


### Findings: income gap by sex

**Raw gap:** the `>50K` rate is 30.38% for men and 10.93% for women. This is a gap of 19.45
percentage points, or a ratio of 2.78 to 1. The association is significant (chi-square
p < 0.001) with a Cramér's V of 0.215, a weak-to-moderate effect.

**Controlled comparisons:**

| Control variable | Gap remaining | % of raw gap | Does it explain the gap? |
|---|---|---|---|
| hours worked | 15.25 points | 78.4% | Barely. Men average 42.4 hours against 36.4 for women, but holding hours constant removes only about a fifth of the gap. |
| occupation | 19.34 points | 99.4% | No. The gap is reproduced almost exactly inside occupations, spread across nearly all of them rather than concentrated in a few. |
| relationship (small groups excluded) | 3.86 points | 19.9% | Mostly. About four fifths of the gap disappears once household role is held constant. |

**A method note, recorded because the first attempt was wrong:** the `relationship` control
first returned about 100% of the gap remaining, which appeared to show that household role
explained nothing. That result was an artefact. The `Husband` and `Wife` categories cannot
support a male–female comparison, because each holds almost no members of one sex (1 woman
recorded as `Husband`, 3 men as `Wife`). Group sizes must therefore be checked before a
stratified comparison is trusted, since a near-empty level can dominate a weighted result.

**Interpretation, recorded as open rather than settled:** the sex income gap overlaps far
more with household and marital role than with hours worked or occupation. This analysis
cannot separate two explanations: that household role itself affects earnings, or that it
records historical expectations about which sex was employed. Both remain consistent with
the data, so the ambiguity is stated as a limitation rather than resolved here.

## 5. Income gap by race

Same approach. Race has five categories, so whether they form distinct groups is answered statistically rather than by inspection.

In [20]:
print(group_rates(df_uci, "race", "income", POSITIVE).sort_values(ascending=False).round(2))
print()
print(df_uci["race"].value_counts())

race
Asian-Pac-Islander    26.93
White                 25.40
Other                 12.32
Black                 12.08
Amer-Indian-Eskimo    11.70
Name: >50K, dtype: float64

race
White                 41762
Black                  4685
Asian-Pac-Islander     1519
Amer-Indian-Eskimo      470
Other                   406
Name: count, dtype: int64


Pairwise two-proportion z-tests across all 10 category pairs, two groups compared at a time on a binary outcome.

In [21]:
race_tests_uci = pairwise_proportion_tests(df_uci, "race", "income", POSITIVE)
race_tests_uci

,group_1,rate_1_pct,n_1,group_2,rate_2_pct,n_2,p_value,verdict
0,Asian-Pac-Islander,26.93,1519,Black,12.08,4685,0.000000e+00,DIFFERENT
1,Black,12.08,4685,White,25.40,41762,0.000000e+00,DIFFERENT
2,Amer-Indian-Eskimo,11.70,470,Asian-Pac-Islander,26.93,1519,9.123147e-12,DIFFERENT
3,Amer-Indian-Eskimo,11.70,470,White,25.40,41762,1.068434e-11,DIFFERENT
4,Asian-Pac-Islander,26.93,1519,Other,12.32,406,8.418353e-10,DIFFERENT
5,Other,12.32,406,White,25.40,41762,1.570518e-09,DIFFERENT
6,Asian-Pac-Islander,26.93,1519,White,25.40,41762,1.795915e-01,same
7,Amer-Indian-Eskimo,11.70,470,Other,12.32,406,7.805405e-01,same
8,Amer-Indian-Eskimo,11.70,470,Black,12.08,4685,8.098416e-01,same
9,Black,12.08,4685,Other,12.32,406,8.896189e-01,same


### Findings: income gap by race

**Raw `>50K` rates:**

| Race | % earning >50K | n |
|---|---|---|
| Asian-Pac-Islander | 26.93% | 1,519 |
| White | 25.40% | 41,762 |
| Other | 12.32% | 406 |
| Black | 12.08% | 4,685 |
| Amer-Indian-Eskimo | 11.70% | 470 |

**Two-cluster structure:** the pairwise tests show two distinct clusters rather than a
gradient or a clean majority-against-minority split.

- Cluster A, higher rates: `White` and `Asian-Pac-Islander`, statistically indistinguishable
  from each other (p = 0.18).
- Cluster B, lower rates: `Black`, `Amer-Indian-Eskimo` and `Other`. All three pairs within
  this cluster are statistically indistinguishable (p = 0.81, 0.78, 0.89).
- Every cross-cluster pair is significantly different (p < 0.0001).

**Caveat:** indistinguishable income outcomes do not mean the groups are socially or
historically similar. `Asian-Pac-Islander` and `White` reach comparable income rates by very
different routes as the native-country finding above suggests and the same caution
applies within Cluster B. The clustering is a statement about one outcome variable, not about
group similarity.

## 6. Race binarisation

Disparate Impact and the other two-group fairness metrics need a binary grouping, so a
binarisation decision cannot be avoided. Two candidate groupings are computed below so that
the difference between them can be measured rather than assumed.

- **Standard grouping:** `White` against all other categories combined. This follows the
  convention used in earlier fairness work on this dataset.
- **Cluster grouping:** `{White, Asian-Pac-Islander}` against
  `{Black, Amer-Indian-Eskimo, Other}`, following the z-test result in Section 5.

**Decision:** the standard grouping is used for the fairness metrics reported in the
dissertation. The cluster grouping is not adopted, because defining the protected groups
using the same income variable that the fairness metrics then measure would be circular. The
cluster finding is kept as a documented limitation of binary group-fairness metrics, which
is the subgroup-masking problem described as fairness gerrymandering by Kearns et al. (2018).

In [22]:
df_uci["race_binary"] = np.where(df_uci["race"] == "White", "White", "Non-White")
df_uci["race_cluster"] = np.where(
    df_uci["race"].isin(["White", "Asian-Pac-Islander"]), "Cluster A", "Cluster B"
)

binarisation_uci = {}
for col in ["race_binary", "race_cluster"]:
    rates = group_rates(df_uci, col, "income", POSITIVE)
    high, low = rates.idxmax(), rates.idxmin()
    binarisation_uci[col] = {
        "advantaged": high,
        "advantaged_rate_pct": round(rates[high], 2),
        "disadvantaged": low,
        "disadvantaged_rate_pct": round(rates[low], 2),
        "gap_pct_points": round(rates[high] - rates[low], 2),
        "ratio": round(rates[low] / rates[high], 3),
    }

pd.DataFrame(binarisation_uci).T

,advantaged,advantaged_rate_pct,disadvantaged,disadvantaged_rate_pct,gap_pct_points,ratio
race_binary,White,25.4,Non-White,15.25,10.14,0.601
race_cluster,Cluster A,25.45,Cluster B,12.07,13.39,0.474


### Findings: race binarisation

Both groupings produce a large disparity and the choice between them does not change the
direction of the result. The ratio column is the base-rate version of Disparate Impact,
measured on the raw data rather than on model predictions. Both groupings fall well below the
0.8 four-fifths threshold before any model has been trained.

`race_binary` is carried forward as the protected group variable for race.

## 7. Export

Summary tables are written to `results/tables/` so that the figures quoted in the
dissertation can be traced back to a file rather than re-derived by hand. A `dataset` column
is added to every table so that the ACS tables can be stacked against these ones in the
comparison notebook.

In [23]:
proxy_table = pd.DataFrame({"cramers_v_sex": proxy_sex, "cramers_v_race": proxy_race})
proxy_table["dataset"] = DATASET
proxy_table.to_csv(f"{RESULTS_DIR}/uci_proxy_strength.csv")

controls_table = pd.DataFrame({
    "dataset": DATASET,
    "control": ["none (raw)", "hours worked", "occupation", "relationship (small groups excluded)"],
    "gap_pct_points": [raw_gap_sex, gap_hours, gap_occ, gap_rel_fixed],
})
controls_table["pct_of_raw_gap"] = controls_table["gap_pct_points"] / raw_gap_sex * 100
controls_table.round(2).to_csv(f"{RESULTS_DIR}/uci_sex_gap_controls.csv", index=False)

race_tests_out = race_tests_uci.copy()
race_tests_out["dataset"] = DATASET
race_tests_out.to_csv(f"{RESULTS_DIR}/uci_race_pairwise_tests.csv", index=False)

binarisation_out = pd.DataFrame(binarisation_uci).T
binarisation_out["dataset"] = DATASET
binarisation_out.to_csv(f"{RESULTS_DIR}/uci_race_binarisation.csv")

print(sorted(os.listdir(RESULTS_DIR)))

['acs_ci_baseline.csv', 'acs_intersectional.csv', 'acs_intersectional_matched.csv', 'acs_model_results.csv', 'acs_multiseed_results.csv', 'acs_multiseed_summary.csv', 'acs_proxy_strength.csv', 'acs_race_binarisation.csv', 'acs_race_pairwise_tests.csv', 'acs_sex_gap_controls.csv', 'acs_threshold_results.csv', 'acs_threshold_sensitivity.csv', 'intersectional_matched_comparison.csv', 'uci_ci_baseline.csv', 'uci_intersectional.csv', 'uci_model_results.csv', 'uci_proxy_strength.csv', 'uci_race_binarisation.csv', 'uci_race_pairwise_tests.csv', 'uci_sex_gap_controls.csv']


In [24]:
# The prepared frame is cached so that the modelling notebook does not need to pull the
# data again. The data/ directory is gitignored, so this copy stays local.
os.makedirs("../data/processed", exist_ok=True)
df_uci.to_csv("../data/processed/uci_1994_prepared.csv", index=False)

print("saved:", "../data/processed/uci_1994_prepared.csv", "| shape:", df_uci.shape)

saved: ../data/processed/uci_1994_prepared.csv | shape: (48842, 18)


## 8. Summary

**Dataset:** 48,842 records and 15 columns. The target is imbalanced at 76.1% against 23.9%.

**Missing values:** found in `workclass` (5.73%), `occupation` (5.75%) and `native-country`
(1.75%) only. The two employment columns are missing together for the same records, and
missingness in `native-country` varies significantly by race. Missing values are kept as an
explicit `"Unknown"` category, since dropping them would discard the records most relevant
to a fairness audit.

**Proxy attributes:** `relationship` is a strong proxy for `sex` (Cramér's V = 0.647) and
`native-country` is a moderate proxy for `race` (Cramér's V = 0.415). Both are removed
alongside the protected attribute in the protected-attribute-removed experiment.

**Sex gap:** 30.38% against 10.93%, a gap of 19.45 percentage points and a ratio of 2.78.
Hours worked explains about a fifth of it and occupation almost none. Household role accounts
for about four fifths, although whether this is a genuine explanation or a restatement of
historical sex roles cannot be settled from this data.

**Race gap:** two distinct clusters are confirmed by pairwise z-tests, with `White` and
`Asian-Pac-Islander` at 25–27% and the remaining three groups at about 12%. Race is binarised
as `White` against `Non-White` for the fairness metrics, with the cluster structure kept as a
documented limitation.

**Carried forward to modelling:** `race_binary` as the protected group variable for race,
`sex` unchanged, and `relationship` plus `native-country` flagged as the proxy features to be
removed in the second experiment.